# Notebook \#3: Modelling

# Eviny Charging Curve Classification LSTM 
#### by Sebastian Einar Salas Røkholt
----

### Index
**01 - Setup** </br>
**02 - Data Exploration, Wrangling and Preprocessing**</br>
**03 - Modelling**</br>
**04 - Model Evaluation and Selection**</br>
**+++**</br>

## 01 - Setup

In [ ]:
import os
import math
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Sampler
import torch.nn.utils.rnn as rnn_utils
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit
from typing import Tuple, Union

RANDOM_SEED = 42
TRAIN_MODEL = True
MODEL_PATH = '../Models/LSTM_varseq_model_2.pth'

# Notebook settings
%matplotlib inline
pd.options.mode.copy_on_write = True
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.options.display.float_format = '{:.2f}'.format  # By default, display all floats with two decimals

In [ ]:
# Loads the cleaned dataset with engineered features (minutes_elapsed, temp)
df = pd.read_parquet("../Data/etron55-charging-sessions.parquet")
# These are the features we will be using throughout the notebook
all_features = ["charging_id", "minutes_elapsed", "power", "soc", "temp", "nominal_power"]
fixed_features = ["temp", "nominal_power"]
target_features = ["power", "soc"]
input_features = fixed_features + target_features

# Removes redundant features such as 'energy', 'charging_duration', 
# 'lat' and 'lon' which we don't need for modelling or plotting results
df = df[all_features]
df.head(10)

In [ ]:
df.tail()

## 02 - Preparing the dataset for modelling

#### Splitting the data 
The code below splits the dataset into training, validation and test sets. </br>
`GroupShuffleSplit` ensures that a charging session isn't split across multiple sets.

In [22]:
def split_data(df: pd.DataFrame, test_size: float=0.2, validation_size: float=0.1) \
    -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split the dataset into train, validation, and test sets by charging_id grouping.
    
    :param df: Original dataframe containing all charging sessions
    :param test_size: Proportion of data to use for testing
    :param validation_size: Proportion of the remaining data (after test split) to use for validation
    :return: train_df, val_df, test_df DataFrames
    """
    # Defines the test split
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_SEED)
    train_val_idx, test_idx = next(gss_test.split(df, groups=df['charging_id']))
    train_val_df = df.iloc[train_val_idx]

    # Defines the validation split
    validation_size = validation_size / (1 - test_size)  # Adjust validation size based on the remaining dataset after test split
    gss_val = GroupShuffleSplit(n_splits=1, test_size=validation_size, random_state=RANDOM_SEED)
    train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['charging_id']))

    # Performs the splits on the original dataset
    train_df = train_val_df.iloc[train_idx]
    val_df = train_val_df.iloc[val_idx]
    test_df = df.iloc[test_idx]
    return train_df, val_df, test_df

train_df, val_df, test_df = split_data(df)

In [ ]:
filtered_df = df.groupby('charging_id').filter(lambda x: x['minutes_elapsed'].max() <= 2)
filtered_df

#### Data normalisation


In [4]:
# Creates scalers
fixed_features_scaler = MinMaxScaler(feature_range=(0, 1))
power_scaler = MinMaxScaler(feature_range=(0, 1))
soc_scaler = MinMaxScaler(feature_range=(0, 1))

# Fits scalers on training data only
fixed_features_scaler.fit(train_df[fixed_features])
power_scaler.fit(train_df[["power"]])
soc_scaler.fit(train_df[["soc"]])

# Transform train, val, test sets
train_df[fixed_features] = fixed_features_scaler.transform(train_df[fixed_features])
val_df[fixed_features] = fixed_features_scaler.transform(val_df[fixed_features])
test_df[fixed_features] = fixed_features_scaler.transform(test_df[fixed_features])
train_df["power"] = power_scaler.transform(train_df[["power"]])
val_df["power"] = power_scaler.transform(val_df[["power"]])
test_df["power"] = power_scaler.transform(test_df[["power"]])
train_df["soc"] = soc_scaler.transform(train_df[["soc"]])
val_df["soc"] = soc_scaler.transform(val_df[["soc"]])
test_df["soc"] = soc_scaler.transform(test_df[["soc"]])

In [5]:
class ChargingSessionDataset(Dataset):
    """
    Returns entire charging sessions as (X, y, length).
    
    X shape: (session_length, input_size)
    y shape: (session_length, output_size)
    length:  scalar int describing actual length
    """
    def __init__(
        self, 
        df: pd.DataFrame,
        input_features,
        target_features,
        shift_next_step=True
    ):
        """
        shift_next_step: if True, we do next-step labeling so that X[t] -> y[t+1].
                         That means we drop the last row from X and the first from y.
        """
        super().__init__()
        self.input_features = input_features
        self.target_features = target_features

        self.sessions = []
        # Group by 'charging_id'
        for session_id, session_df in df.groupby('charging_id', observed=False):
            session_df = session_df.sort_values('minutes_elapsed')
            # Convert to numpy
            session_x = session_df[self.input_features].values
            session_y = session_df[self.target_features].values

            # SHIFT for "predict all timesteps > 0"
            if shift_next_step and len(session_x) > 1:
                # X: [0..n-2], y: [1..n-1]
                session_x = session_x[:-1]
                session_y = session_y[1:]

            if len(session_x) > 0:
                self.sessions.append((session_x, session_y))

    def __len__(self):
        return len(self.sessions)

    def __getitem__(self, idx):
        x, y = self.sessions[idx]
        length = x.shape[0]
        return x, y, length


class BucketBatchSampler(Sampler):
    """
    Groups sessions of similar length into the same batch to minimize padding.
    1) Sort all session indices by ascending length.
    2) Divide them into chunks of size `batch_size`.
    3) Shuffle the chunks themselves (optional).
    4) Yield each chunk as one batch of indices.
    """
    def __init__(self, dataset: ChargingSessionDataset, batch_size: int, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        
        # Sort dataset indices by ascending session length
        lengths = []
        for (i, (x, y)) in enumerate(dataset.sessions):
            lengths.append((i, x.shape[0]))
        # Sort by length
        self.sorted_indices = sorted(lengths, key=lambda tup: tup[1])
        # We only need the indices
        self.sorted_indices = [tup[0] for tup in self.sorted_indices]

        # Chunk them into batches
        self.batches = []
        for i in range(0, len(self.sorted_indices), self.batch_size):
            self.batches.append(self.sorted_indices[i : i + self.batch_size])
        
        # Shuffle the list of batches if desired
        if self.shuffle:
            random.shuffle(self.batches)

    def __iter__(self):
        # Each epoch, optionally re-shuffle the batch order
        if self.shuffle:
            random.shuffle(self.batches)
        for batch_inds in self.batches:
            yield batch_inds

    def __len__(self):
        return math.ceil(len(self.sorted_indices) / self.batch_size)


def charging_collate_fn(batch):
    all_x, all_y, lengths = zip(*batch)
    max_len = max(lengths)

    padded_x = []
    padded_y = []
    for x, y in zip(all_x, all_y):
        seq_len = x.shape[0]
        
        x_padded = np.zeros((max_len, x.shape[1]), dtype=np.float32)
        x_padded[:seq_len, :] = x
        
        y_padded = np.zeros((max_len, y.shape[1]), dtype=np.float32)
        y_padded[:seq_len, :] = y

        padded_x.append(x_padded)
        padded_y.append(y_padded)

    # Convert list of ndarrays -> single ndarray -> tensor
    padded_x = torch.from_numpy(np.stack(padded_x, axis=0)).float()
    padded_y = torch.from_numpy(np.stack(padded_y, axis=0)).float()
    lengths  = torch.tensor(lengths, dtype=torch.long)

    return padded_x, padded_y, lengths


In [6]:
# Instead of create_sequences(), we now build entire-session datasets:
train_dataset = ChargingSessionDataset(train_df, input_features, target_features, shift_next_step=True)
val_dataset   = ChargingSessionDataset(val_df,   input_features, target_features, shift_next_step=True)
test_dataset  = ChargingSessionDataset(test_df,  input_features, target_features, shift_next_step=True)

# Bucket-based sampling
BATCH_SIZE = 128
NUM_WORKERS = 8

train_sampler = BucketBatchSampler(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_sampler   = BucketBatchSampler(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_sampler=train_sampler,
    collate_fn=charging_collate_fn,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_sampler=val_sampler,
    collate_fn=charging_collate_fn,
    num_workers=NUM_WORKERS
)

## Modelling

In [7]:
class MultivariateLSTM(nn.Module):
    """
    LSTM that predicts multiple outputs for each valid time step.
    """
    def __init__(self, input_size, hidden_layer_size, output_size, num_layers):
        super(MultivariateLSTM, self).__init__()
        self.hidden_layer_size = hidden_layer_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size, 
            hidden_size=hidden_layer_size, 
            num_layers=num_layers, 
            batch_first=True
        )
        self.linear = nn.Linear(hidden_layer_size, output_size)

    def forward(self, x, seq_lengths):
        """
        :param x: Padded input tensor of shape (batch_size, max_seq_len, input_size)
        :param seq_lengths: 1D tensor with actual lengths (batch_size,)
        :return:
          out: Padded output of shape (batch_size, max_seq_len, output_size)
          out_lengths: same as seq_lengths (for completeness)
        """
        # Pack the padded sequence so LSTM ignores the padded steps
        packed_x = rnn_utils.pack_padded_sequence(
            x, 
            seq_lengths.cpu(),  # lengths must be on CPU for pack_padded_sequence
            batch_first=True, 
            enforce_sorted=False
        )
        
        # Forward through LSTM
        packed_out, (h, c) = self.lstm(packed_x)
        
        # Unpack so we get a padded output of shape (B, T_max, hidden_layer_size)
        out, out_lengths = rnn_utils.pad_packed_sequence(packed_out, batch_first=True)
        
        # Apply the final linear layer at every time step
        out = self.linear(out)  # shape: (batch_size, T_max, output_size)
        
        return out, out_lengths

In [8]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 10,
    initial_lr: float = 0.001,
    plot_loss: bool = True
) -> None:
    """
    Train the given PyTorch model using the provided data loaders,
    ignoring padded timesteps in the loss function.
    """    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    # We'll use MSELoss with no reduction, so we can mask out padded steps
    criterion = nn.MSELoss(reduction='none')
    optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    if plot_loss:
        train_losses, val_losses = [], []
    
    for epoch in range(num_epochs):
        ############################
        # Training
        ############################
        model.train()
        epoch_train_loss = 0.0
        total_train_valid_steps = 0  # number of valid (non-padded) timesteps

        for X_batch, Y_batch, lengths in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
            lengths = lengths.to(device)

            optimizer.zero_grad()

            # Forward
            pred_padded, out_lengths = model(X_batch, lengths)
            # pred_padded -> [B, T_max, output_size]
            # Y_batch     -> [B, T_max, output_size]

            # We'll create a mask to ignore padded steps
            B, T_max, _ = pred_padded.shape
            mask = torch.zeros(B, T_max, dtype=torch.bool, device=device)
            for i in range(B):
                mask[i, :lengths[i]] = True

            # MSE per element
            loss_per_timestep = criterion(pred_padded, Y_batch)  # [B, T_max, output_size]

            # Sum over features => [B, T_max]
            loss_per_timestep = loss_per_timestep.sum(dim=-1)

            # Keep only valid timesteps
            valid_loss = loss_per_timestep[mask].sum()
            valid_steps = mask.sum().item()  # total non-padded steps

            # Mean MSE over valid steps
            loss = valid_loss / valid_steps

            loss.backward()
            optimizer.step()

            epoch_train_loss += valid_loss.item()  # keep sum of MSE
            total_train_valid_steps += valid_steps

        avg_epoch_train_loss = epoch_train_loss / total_train_valid_steps

        ############################
        # Validation
        ############################
        model.eval()
        epoch_val_loss = 0.0
        total_val_valid_steps = 0

        with torch.no_grad():
            for X_batch, Y_batch, lengths in val_loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)
                lengths = lengths.to(device)

                pred_padded, out_lengths = model(X_batch, lengths)
                
                B, T_max, _ = pred_padded.shape
                mask = torch.zeros(B, T_max, dtype=torch.bool, device=device)
                for i in range(B):
                    mask[i, :lengths[i]] = True

                loss_per_timestep = criterion(pred_padded, Y_batch)  # [B, T_max, output_size]
                loss_per_timestep = loss_per_timestep.sum(dim=-1)
                
                valid_loss = loss_per_timestep[mask].sum()
                valid_steps = mask.sum().item()
                
                epoch_val_loss += valid_loss.item()
                total_val_valid_steps += valid_steps

        avg_epoch_val_loss = epoch_val_loss / total_val_valid_steps

        # Scheduler step
        scheduler.step(avg_epoch_val_loss)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, LR={scheduler.get_last_lr()[0]:.3e}, "
                  f"TrainLoss={avg_epoch_train_loss:.6f}, ValLoss={avg_epoch_val_loss:.6f}")

        if plot_loss:
            train_losses.append(avg_epoch_train_loss)
            val_losses.append(avg_epoch_val_loss)

    print('Training complete.')

    if plot_loss:
        # Plot training vs validation loss
        plt.figure(figsize=(8,5))
        plt.plot(range(1, len(train_losses)+1), train_losses, label='Train')
        plt.plot(range(1, len(val_losses)+1), val_losses, label='Val')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.title('Training & Validation MSE (per valid timestep)')
        plt.show()


In [ ]:
# Setting the model hyperparameters
input_size = len(input_features)  # power, soc, minutes_elapsed, temperature and nominal power
output_size = len(target_features)  # Predicting power and soc for the next time step
HIDDEN_LAYER_SIZE = 60
NUM_HIDDEN_LAYERS = 4
N_TRAINING_EPOCHS = 5

# Defining the network architecture and hyperparameters
model = MultivariateLSTM(input_size, HIDDEN_LAYER_SIZE, output_size, NUM_HIDDEN_LAYERS)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Running the training loop
if TRAIN_MODEL:
    print(f"Training on device {device}...")
    train_model(model, train_loader, val_loader, num_epochs=N_TRAINING_EPOCHS, plot_loss=True)
    torch.save(model.state_dict(), MODEL_PATH)
elif not TRAIN_MODEL and os.path.exists(MODEL_PATH): 
    # Load the model
    model.load_state_dict(torch.load(MODEL_PATH))
    model.eval()
else:
    print(f"TRAIN_MODEL is set to False, but there is no pre-trained model saved at path {MODEL_PATH}")
    print(f"Training a new model on device {device}...")
    train_model(model, train_loader, val_loader, num_epochs=10, plot_loss=True)
    torch.save(model.state_dict(), MODEL_PATH)

## Prediction

In [11]:
# For test, we typically also want a DataLoader:
test_sampler = BucketBatchSampler(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(
    dataset=test_dataset,
    batch_sampler=test_sampler,
    collate_fn=charging_collate_fn,
    num_workers=NUM_WORKERS
)

In [ ]:
# Here's an example of running inference on the *test* set. For demonstration, we'll pick
# one test batch and see how well it reconstructs (power, soc) across time steps.

# %%
test_batch = next(iter(test_loader))  # X_batch, Y_batch, lengths
X_test_batch, Y_test_batch, lengths_test = test_batch
print("Batch lengths:", lengths_test)
X_test_batch = X_test_batch.to(device)
Y_test_batch = Y_test_batch.to(device)
lengths_test = lengths_test.to(device)

model.eval()
with torch.no_grad():
    y_pred_padded, out_lengths = model(X_test_batch, lengths_test)
    print(y_pred_padded)
# y_pred_padded: shape [B, T_max, output_size]
# We'll invert scaling for a single item as an example:

i_sample = -1
sample_len = lengths_test[i_sample].item()
pred_np = y_pred_padded[i_sample, :sample_len].cpu().numpy()
true_np = Y_test_batch[i_sample, :sample_len].cpu().numpy()
print(i_sample, "\n")
print(pred_np, "\n____________\n")
print(true_np)
# Inverse transform the columns
pred_power = power_scaler.inverse_transform(pred_np[:, [0]]).ravel()
pred_soc   = soc_scaler.inverse_transform(pred_np[:, [1]]).ravel()
true_power = power_scaler.inverse_transform(true_np[:, [0]]).ravel()
true_soc   = soc_scaler.inverse_transform(true_np[:, [1]]).ravel()

# Just a quick plot
time_axis = np.arange(sample_len)  # or your original minutes_elapsed if you stored them
plt.figure(figsize=(10,4))
plt.plot(time_axis, true_power, label='True Power', color='b')
plt.plot(time_axis, pred_power, label='Pred Power', color='r', linestyle='--')
plt.xlabel('Timestep')
plt.ylabel('Power (unscaled)')
plt.title('Test Sample #0 Power')
plt.legend()
plt.show()

plt.figure(figsize=(10,4))
plt.plot(time_axis, true_soc, label='True SOC', color='b')
plt.plot(time_axis, pred_soc, label='Pred SOC', color='r', linestyle='--')
plt.xlabel('Timestep')
plt.ylabel('SOC (unscaled)')
plt.title('Test Sample #0 SOC')
plt.legend()
plt.show()

In [ ]:
for idx in range(len(test_dataset)):
    X_arr, Y_arr, length_ = test_dataset[idx]
    print(f"test_dataset[{idx}]: length={length_}, X.shape={X_arr.shape}, Y.shape={Y_arr.shape}")


In [ ]:
print("Lengths in test batch:", lengths_test)